# Phase 2: SQL Analysis
In this notebook we load the cleaned finance data into a SQLite database
and write queries to answer key business questions about spending and income patterns. 

In [4]:
import pandas as pd 
import sqlite3

#Load cleaned data
df = pd.read_csv('../data/cleaned_finance_data.csv')

# Create a SQLite database in memory 
conn = sqlite3.connect(':memory:')

# Load data frame into SQLite as table called 'transactions'
df.to_sql('transactions', conn, index=False, if_exists='replace')

print("Database loaded successfully!")
print("Rows loaded:", len(df))


Database loaded successfully!
Rows loaded: 1500


## Business Questions
1. What is the total income vs total expenses?
2. What category has the highest spending?
3. What is the monthly spending trend?
4. What is the average transaction amount by category?
5. Which months had the highest income vs expenses?

In [5]:
query = """
SELECT
    Type,
    Round(SUM(Amount), 2) AS Total_Amount, 
    COUNT(*) AS Transaction_Count
FROM transactions
GROUP BY Type
"""

result = pd.read_sql_query(query, conn)
print(result)

      Type  Total_Amount  Transaction_Count
0  Expense    1227194.37               1222
1   Income     734087.00                278


In [6]:
query = """
SELECT 
    Category,
    ROUND(SUM(Amount), 2) AS Total_Spent,
    COUNT(*) AS Number_of_Transactions,
    ROUND(AVG(Amount), 2) AS Avg_Transaction
FROM transactions
WHERE Type = 'Expense'
GROUP BY Category
ORDER BY Total_Spent DESC
"""

result = pd.read_sql_query(query, conn)
print(result)

           Category  Total_Spent  Number_of_Transactions  Avg_Transaction
0            Travel    169497.79                     160          1059.36
1              Rent    162075.39                     165           982.28
2      Food & Drink    159493.39                     149          1070.43
3            Salary    149053.55                     146          1020.91
4     Entertainment    148165.47                     143          1036.12
5          Shopping    146880.75                     150           979.21
6         Utilities    146833.97                     157           935.25
7  Health & Fitness    145194.06                     152           955.22


In [7]:
query = """
SELECT 
    Year,
    Month,
    Month_Name,
    Type,
    ROUND(SUM(Amount), 2) AS Total_Amount
FROM transactions
GROUP BY Year, Month, Month_Name, Type
ORDER BY Year, Month
"""

result = pd.read_sql_query(query, conn)
print(result)

     Year  Month Month_Name     Type  Total_Amount
0    2020      1    January  Expense      17138.25
1    2020      1    January   Income       5578.00
2    2020      2   February  Expense      17108.41
3    2020      2   February   Income      20070.00
4    2020      3      March  Expense      13581.81
..    ...    ...        ...      ...           ...
115  2024     10    October   Income      12974.00
116  2024     11   November  Expense      23565.96
117  2024     11   November   Income      24139.00
118  2024     12   December  Expense      10938.92
119  2024     12   December   Income       6530.00

[120 rows x 5 columns]


In [8]:
query = """
SELECT 
    Category,
    ROUND(AVG(Amount), 2) AS Avg_Amount,
    ROUND(MIN(Amount), 2) AS Min_Amount,
    ROUND(MAX(Amount), 2) AS Max_Amount
FROM transactions
GROUP BY Category
ORDER BY Avg_Amount DESC
"""

result = pd.read_sql_query(query, conn)
print(result)

           Category  Avg_Amount  Min_Amount  Max_Amount
0             Other     2726.73      538.00     4975.00
1        Investment     2558.11      500.00     4996.00
2      Food & Drink     1070.43       29.25     1999.82
3            Travel     1059.36       53.66     1994.88
4     Entertainment     1036.12       14.37     1980.56
5            Salary     1020.91       28.48     1993.30
6              Rent      982.28       25.38     1976.43
7          Shopping      979.21       19.29     1962.27
8  Health & Fitness      955.22       46.28     1996.26
9         Utilities      935.25       15.32     1985.77


## Key Findings
- Total expenses ($1,227,194) significantly exceed total income ($734,087)
- Travel is the highest spending category at $169,497 across 160 transactions
- Other and Investment categories have the highest average transaction values
- Monthly trends show variability in both income and expenses across 2020-2024

**Data is ready for visualization in Phase 3.**

In [9]:
# Save monthly trends for visualization
monthly = pd.read_sql_query("""
    SELECT Year, Month, Month_Name, Type, ROUND(SUM(Amount), 2) AS Total_Amount
    FROM transactions
    GROUP BY Year, Month, Month_Name, Type
    ORDER BY Year, Month
""", conn)

# Save category spending for visualization
category_spend = pd.read_sql_query("""
    SELECT Category, ROUND(SUM(Amount), 2) AS Total_Spent
    FROM transactions
    WHERE Type = 'Expense'
    GROUP BY Category
    ORDER BY Total_Spent DESC
""", conn)

monthly.to_csv('../data/monthly_trends.csv', index=False)
category_spend.to_csv('../data/category_spend.csv', index=False)

print("Query results saved successfully!")

Query results saved successfully!
